# 🤝 SAATHI AI - 1-Click Model Fine-Tuning (Google Colab Free GPU)
> **Fine-tune your custom Indian Multilingual Assistant Model using QLoRA**

This notebook lets you fine-tune open-source models like `Qwen/Qwen2.5-1.5B-Instruct` or `meta-llama/Llama-3.2-1B-Instruct` on the SAATHI dataset using a free Google Colab T4 GPU.

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install required ML libraries in Colab
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl

In [ ]:
# Step 3: Write SAATHI Training Script and Dataset
%%writefile train_saathi_colab.py
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# 1. Create Training Dataset
training_samples = [
    {
        "instruction": "You are SAATHI AI. Classify user query: [DIRECT_ANSWER, RETRIEVE_AND_ANSWER, CLARIFY, PREPARE_ACTION, ESCALATE].",
        "input": "Mera bijli ka bill bahut jyada aaya hai meter kharab lagta hai",
        "output": "DECISION: RETRIEVE_AND_ANSWER\nREASONING: Factual query about electricity bill and faulty meter procedure."
    },
    {
        "instruction": "Explain how PM-KISAN Samman Nidhi installment errors (like land seeding or eKYC) can be fixed.",
        "input": "PM kisan ka 2000 rupaye nahi aaya, land seeding NO dikha raha hai, kya karein?",
        "output": "Agar PM-KISAN portal par Land Seeding 'NO' hai, toh: 1) Khatoni/Bhulekh aur Aadhaar Patwari ke paas submit karein. 2) e-KYC complete karein. 3) Bank me NPCI Aadhaar seeding verify karein."
    },
    {
        "instruction": "Explain how to report cyber financial fraud and recover stolen money.",
        "input": "Online fraud ho gaya paise kat gaye, helpline number kya hai?",
        "output": "1) Turant National Cyber Crime Helpline '1930' par call karein. 2) cybercrime.gov.in par complaint darj karein. 3) Bank ko 3 din ke andar inform karein."
    }
]

with open("saathi_train.jsonl", "w", encoding="utf-8") as f:
    for s in training_samples:
        f.write(json.dumps(s) + "\n")

print("✅ Dataset generated: saathi_train.jsonl")

# 2. Load Base Model and Tokenizer
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. Format Prompts and Tokenize
def format_prompts(example):
    formatted = f"<|im_start|>system\nYou are SAATHI AI, a safe assistant for Indian daily life.<|im_end|>\n<|im_start|>user\n{example['instruction']}\n\nInput: {example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"
    tokens = tokenizer(formatted, max_length=512, truncation=True, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

raw_data = load_dataset("json", data_files={"train": "saathi_train.jsonl"})
tokenized_data = raw_data["train"].map(format_prompts)

# 5. Train Model
training_args = TrainingArguments(
    output_dir="saathi_lora_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data
)
print("🏋️ Starting QLoRA Training on Colab GPU...")
trainer.train()
model.save_pretrained("saathi_lora_adapter")
tokenizer.save_pretrained("saathi_lora_adapter")
print("🎉 Training Complete! Adapter saved to saathi_lora_adapter/")


In [ ]:
# Step 4: Execute Training on Colab GPU
!python train_saathi_colab.py

In [ ]:
# Step 5: Test the Trained Model
%%writefile test_saathi_colab.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_dir = "saathi_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_dir)

prompt = "<|im_start|>system\nYou are SAATHI AI.<|im_end|>\n<|im_start|>user\nOnline fraud ho gaya paise kat gaye kya karein?<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
# Step 6: Run Model Inference Test
!python test_saathi_colab.py